# EDS JSON to DataFrame Converter

This notebook reads individual JSON files from the Claude extraction output and creates a single pandas DataFrame with one row per form.

## 1. Setup and Imports

In [75]:
import os
import json
import pandas as pd
from pathlib import Path
from typing import Dict, List, Any

# Configuration - match the output_dir from eds_extraction_claude.ipynb
jloc = '../../../'
JSON_DIR = jloc + 'data/intermediate_products/eds_claude_json_production'

print(f"Reading JSON files from: {JSON_DIR}")

Reading JSON files from: ../../../data/intermediate_products/eds_claude_json_production


## 2. Load and Parse JSON Files

In [77]:
def merge_split_account_numbers(account_numbers: List[str]) -> List[str]:
    """
    Merge account number fragments that were incorrectly split due to OCR errors.
    
    Common issue: ["1210-1-3400-5310", "10"] should be ["1210-1-3400-531010"]
    
    Heuristic: If an element is very short (< 4 chars) and numeric-like,
    it's likely a fragment that should be merged with the previous element.
    
    Args:
        account_numbers: List of account number strings
        
    Returns:
        List of merged account numbers
    """
    if not account_numbers or len(account_numbers) == 0:
        return account_numbers
    
    merged = []
    i = 0
    
    while i < len(account_numbers):
        current = str(account_numbers[i]).strip()
        
        # Check if this looks like a fragment
        # Criteria: short (< 4 chars) and primarily digits/dashes
        is_fragment = (
            len(current) < 4 and 
            i > 0 and  # Not the first element
            current.replace('-', '').replace('.', '').replace(' ', '').isdigit()
        )
        
        if is_fragment and merged:
            # Merge with previous element
            prev = merged[-1]
            # Remove any trailing separator and add the fragment
            if prev.endswith('-') or prev.endswith('.'):
                merged[-1] = prev + current
            else:
                merged[-1] = prev + current
        else:
            # Normal element, add as-is
            merged.append(current)
        
        i += 1
    
    return merged


def flatten_json(json_data: Dict) -> Dict:
    """
    Flatten nested JSON structure into a single-level dictionary.
    Handles metadata, extracted_data, contract_type_info, and checkbox_fields.
    """
    flat = {}
    
    # Extract metadata fields
    if 'metadata' in json_data:
        for key, value in json_data['metadata'].items():
            flat[f'metadata_{key}'] = value
    
    # Extract main extracted_data fields
    if 'extracted_data' in json_data:
        data = json_data['extracted_data']
        
        # Handle main fields (non-nested)
        for key, value in data.items():
            if key not in ['contract_type_info', 'checkbox_fields']:
                # Convert lists to comma-separated strings for dataframe compatibility
                if isinstance(value, list):
                    # Special handling for account_numbers: merge split fragments
                    if key == 'account_numbers':
                        value = merge_split_account_numbers(value)
                    flat[key] = ', '.join(str(v) for v in value) if value else None
                else:
                    flat[key] = value
        
        # Handle contract_type_info
        if 'contract_type_info' in data:
            contract_info = data['contract_type_info']
            if isinstance(contract_info, dict):
                # Convert contract_types array to comma-separated string
                if 'contract_types' in contract_info:
                    types = contract_info['contract_types']
                    flat['contract_types'] = ', '.join(types) if types else None
                
                # Convert contract_confidence array to comma-separated string
                if 'contract_confidence' in contract_info:
                    conf = contract_info['contract_confidence']
                    flat['contract_confidence'] = ', '.join(str(c) for c in conf) if conf else None
                
                # Convert selection_types array to comma-separated string
                if 'selection_types' in contract_info:
                    types = contract_info['selection_types']
                    flat['selection_types'] = ', '.join(types) if types else None
                
                # Convert selection_confidence array to comma-separated string
                if 'selection_confidence' in contract_info:
                    conf = contract_info['selection_confidence']
                    flat['selection_confidence'] = ', '.join(str(c) for c in conf) if conf else None
        
        # Handle checkbox_fields
        if 'checkbox_fields' in data:
            checkbox = data['checkbox_fields']
            if isinstance(checkbox, dict):
                # Top-level checkbox fields
                for key in ['vendor_registered', 'renewal_language', 'termination_clause', 'data_processing_involved']:
                    if key in checkbox:
                        flat[key] = checkbox[key]
                
                # Primary vendor status
                if 'primary_vendor_status' in checkbox:
                    pv = checkbox['primary_vendor_status']
                    if isinstance(pv, dict):
                        flat['primary_vendor_minority'] = pv.get('minority')
                        flat['primary_vendor_women'] = pv.get('women')
                        flat['primary_vendor_in_veteran'] = pv.get('in_veteran')
                
                # Subvendor status
                if 'subvendor_status' in checkbox:
                    sv = checkbox['subvendor_status']
                    if isinstance(sv, dict):
                        flat['subvendor_minority'] = sv.get('minority')
                        flat['subvendor_women'] = sv.get('women')
                        flat['subvendor_in_veteran'] = sv.get('in_veteran')
    
    return flat


def load_all_json_files(json_dir: str) -> List[Dict]:
    """
    Load all JSON files from the directory and return a list of flattened dictionaries.
    """
    json_path = Path(json_dir)
    
    if not json_path.exists():
        print(f"Error: Directory not found: {json_dir}")
        return []
    
    all_data = []
    json_files = list(json_path.glob('*.json'))
    
    # Filter out batch registry and metadata files
    json_files = [f for f in json_files if not f.name.startswith('batch_') or not f.name.endswith('.meta.json')]
    
    print(f"Found {len(json_files)} JSON files to process")
    
    errors = 0
    for json_file in json_files:
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
                flat_data = flatten_json(data)
                # Add filename for reference
                flat_data['json_filename'] = json_file.name
                all_data.append(flat_data)
        except Exception as e:
            print(f"Error loading {json_file.name}: {e}")
            errors += 1
    
    print(f"Successfully loaded {len(all_data)} JSON files")
    if errors > 0:
        print(f"Errors: {errors}")
    
    return all_data


print("Functions defined (with account number split detection).")

Functions defined (with account number split detection).


In [78]:
# Test the account number merging logic
print("Testing account number split detection and merging:")
print("="*60)

test_cases = [
    (["1210-1-3400-5310", "10"], "Split fragment at end"),
    (["12910-F5480.531", "010"], "Fragment with leading zero"),
    (["ACCT-123", "45", "67"], "Multiple fragments"),
    (["VALID-ACCOUNT-NUMBER"], "Normal account (no split)"),
    (["ACCT-1", "ACCT-2"], "Multiple separate accounts"),
]

for test_input, description in test_cases:
    result = merge_split_account_numbers(test_input)
    print(f"\n{description}:")
    print(f"  Input:  {test_input}")
    print(f"  Output: {result}")

Testing account number split detection and merging:

Split fragment at end:
  Input:  ['1210-1-3400-5310', '10']
  Output: ['1210-1-3400-531010']

Fragment with leading zero:
  Input:  ['12910-F5480.531', '010']
  Output: ['12910-F5480.531010']

Multiple fragments:
  Input:  ['ACCT-123', '45', '67']
  Output: ['ACCT-1234567']

Normal account (no split):
  Input:  ['VALID-ACCOUNT-NUMBER']
  Output: ['VALID-ACCOUNT-NUMBER']

Multiple separate accounts:
  Input:  ['ACCT-1', 'ACCT-2']
  Output: ['ACCT-1', 'ACCT-2']


## 3. Create DataFrame

In [80]:
# Load all JSON files
all_data = load_all_json_files(JSON_DIR)

# Create DataFrame
if all_data:
    df = pd.DataFrame(all_data)
    print(f"\nDataFrame created with {len(df)} rows and {len(df.columns)} columns")
    print(f"\nColumn names:")
    for col in sorted(df.columns):
        print(f"  - {col}")
else:
    print("No data loaded. DataFrame not created.")
    df = pd.DataFrame()

Found 26109 JSON files to process
Successfully loaded 26109 JSON files

DataFrame created with 26109 rows and 59 columns

Column names:
  - account_names
  - account_numbers
  - agency_address
  - agency_contact_email
  - agency_contact_name
  - agency_contact_phone
  - agency_name
  - approval_dates
  - approver_names
  - contract_amount_new_total
  - contract_amount_this_action
  - contract_confidence
  - contract_types
  - courier_email
  - courier_name
  - courier_phone
  - data_processing_involved
  - date_prepared
  - description_of work
  - description_of_work
  - eds_number
  - error
  - explanation_if_late
  - fiscal_year
  - fiscal_year_amount
  - fiscal_year_amounts
  - json_filename
  - justification_vendor_selection
  - metadata_extracted_at
  - metadata_model
  - metadata_page_number
  - metadata_source_file
  - method_of_source_selection
  - primary_vendor_in_veteran
  - primary_vendor_minority
  - primary_vendor_women
  - raw_response
  - renewal_language
  - requisitio

## 5. Inspect DataFrame

## 4. Identify and Merge Duplicate Columns

Columns that differ only by spacing or punctuation (e.g., "description_of_work" vs "description_of work") should be merged.

In [83]:
import re
from collections import defaultdict

def normalize_column_name(col_name: str) -> str:
    """
    Normalize column name by removing/replacing punctuation and spacing.
    Converts to lowercase and replaces non-alphanumeric chars with underscores.
    """
    # Convert to lowercase
    normalized = col_name.lower()
    # Replace any non-alphanumeric characters (except underscore) with underscore
    normalized = re.sub(r'[^a-z0-9_]', '_', normalized)
    # Remove leading/trailing underscores
    normalized = normalized.strip('_')
    # Collapse multiple underscores into single underscore
    normalized = re.sub(r'_+', '_', normalized)
    return normalized


def identify_duplicate_columns(df: pd.DataFrame) -> Dict[str, List[str]]:
    """
    Identify groups of columns that normalize to the same value.
    Returns dict mapping normalized_name -> [list of original column names]
    """
    normalized_map = defaultdict(list)
    
    for col in df.columns:
        normalized = normalize_column_name(col)
        normalized_map[normalized].append(col)
    
    # Only return groups with duplicates
    duplicates = {k: v for k, v in normalized_map.items() if len(v) > 1}
    return duplicates


def merge_duplicate_columns(df: pd.DataFrame, duplicate_groups: Dict[str, List[str]]) -> pd.DataFrame:
    """
    Merge duplicate columns by taking the first non-null value.
    Keeps the column name with the most non-null values.
    """
    df_merged = df.copy()
    
    for normalized_name, col_list in duplicate_groups.items():
        print(f"\nMerging columns for '{normalized_name}':")
        
        # Sort by number of non-null values (descending)
        col_counts = [(col, df[col].notna().sum()) for col in col_list]
        col_counts.sort(key=lambda x: x[1], reverse=True)
        
        print(f"  Column variants found:")
        for col, count in col_counts:
            pct = (count / len(df) * 100).round(1)
            print(f"    - '{col}': {count} non-null ({pct}%)")
        
        # Use the column with most non-null values as the primary
        primary_col = col_counts[0][0]
        other_cols = [col for col, _ in col_counts[1:]]
        
        # Create merged column: coalesce across all variants
        # Start with primary column, fill nulls from other columns in order
        merged_series = df[primary_col].copy()
        for other_col in other_cols:
            merged_series = merged_series.fillna(df[other_col])
        
        # Count how many values came from secondary columns
        original_nulls = df[primary_col].isna().sum()
        filled_nulls = merged_series.notna().sum() - df[primary_col].notna().sum()
        
        if filled_nulls > 0:
            print(f"  → Merged {filled_nulls} additional value(s) from secondary column(s)")
        
        # Replace primary column with merged values
        df_merged[primary_col] = merged_series
        
        # Drop the duplicate columns
        df_merged = df_merged.drop(columns=other_cols)
        print(f"  → Kept '{primary_col}' and dropped {len(other_cols)} duplicate(s)")
    
    return df_merged


def merge_specific_columns(df: pd.DataFrame, column_groups: List[List[str]], 
                          primary_cols: List[str] = None) -> pd.DataFrame:
    """
    Merge specific column groups manually specified by the user.
    
    Args:
        df: DataFrame to merge
        column_groups: List of lists, where each inner list contains column names to merge
                      Example: [['description_of_work', 'description_of work'], 
                               ['vendor_id', 'vendor_id_number']]
        primary_cols: Optional list of column names to keep (one per group).
                     If None, keeps the column with most non-null values.
                     
    Returns:
        DataFrame with specified columns merged
    """
    df_merged = df.copy()
    
    for i, col_group in enumerate(column_groups):
        # Validate that all columns exist
        missing_cols = [col for col in col_group if col not in df_merged.columns]
        if missing_cols:
            print(f"\nWarning: Skipping group {i+1} - columns not found: {missing_cols}")
            continue
        
        if len(col_group) < 2:
            print(f"\nWarning: Skipping group {i+1} - need at least 2 columns to merge")
            continue
        
        print(f"\nMerging column group {i+1}:")
        
        # Determine primary column
        if primary_cols and i < len(primary_cols) and primary_cols[i] in col_group:
            primary_col = primary_cols[i]
            other_cols = [col for col in col_group if col != primary_col]
        else:
            # Auto-select column with most non-null values
            col_counts = [(col, df_merged[col].notna().sum()) for col in col_group]
            col_counts.sort(key=lambda x: x[1], reverse=True)
            primary_col = col_counts[0][0]
            other_cols = [col for col, _ in col_counts[1:]]
        
        print(f"  Columns to merge:")
        for col in col_group:
            count = df_merged[col].notna().sum()
            pct = (count / len(df_merged) * 100).round(1)
            marker = " (PRIMARY)" if col == primary_col else ""
            print(f"    - '{col}': {count} non-null ({pct}%){marker}")
        
        # Create merged column
        merged_series = df_merged[primary_col].copy()
        for other_col in other_cols:
            merged_series = merged_series.fillna(df_merged[other_col])
        
        # Count how many values came from secondary columns
        filled_nulls = merged_series.notna().sum() - df_merged[primary_col].notna().sum()
        
        if filled_nulls > 0:
            print(f"  → Merged {filled_nulls} additional value(s) from secondary column(s)")
        else:
            print(f"  → No additional values merged (primary column already complete)")
        
        # Replace primary column with merged values
        df_merged[primary_col] = merged_series
        
        # Drop the duplicate columns
        df_merged = df_merged.drop(columns=other_cols)
        print(f"  → Kept '{primary_col}' and dropped: {', '.join(other_cols)}")
    
    return df_merged


# Run the duplicate detection and merging
if not df.empty:
    print("Checking for duplicate columns...")
    print("=" * 60)
    
    duplicate_groups = identify_duplicate_columns(df)
    
    if duplicate_groups:
        print(f"\nFound {len(duplicate_groups)} group(s) of duplicate columns")
        print("=" * 60)
        
        # Merge duplicates
        df = merge_duplicate_columns(df, duplicate_groups)
        
        print("\n" + "=" * 60)
        print(f"DataFrame after merging: {len(df)} rows, {len(df.columns)} columns")
    else:
        print("\nNo duplicate columns found.")
    
    # Manual merging - customize the column groups below
    manual_merges = [
        ['vendor_id', 'vendor_id_number'],
        ['fiscal_year_amount', 'fiscal_year_amounts'],
        ['revenue_generated', 'revenue_generated_this_action', 'revenue_generated_total_contract'],
    ]
    
    # Optional: specify which columns to keep (same order as manual_merges)
    # If not specified, will auto-select column with most non-null values
    keep_columns = ['vendor_id', 'fiscal_year_amount', 'revenue_generated']
    
    print("\n" + "=" * 60)
    print("MANUAL COLUMN MERGING")
    print("=" * 60)
    df = merge_specific_columns(df, manual_merges, primary_cols=keep_columns)
    print("\n" + "=" * 60)
    print(f"DataFrame after manual merging: {len(df)} rows, {len(df.columns)} columns")
    
else:
    print("DataFrame is empty")

Checking for duplicate columns...

Found 1 group(s) of duplicate columns

Merging columns for 'description_of_work':
  Column variants found:
    - 'description_of_work': 26042 non-null (99.7%)
    - 'description_of work': 3 non-null (0.0%)
  → Merged 3 additional value(s) from secondary column(s)
  → Kept 'description_of_work' and dropped 1 duplicate(s)

DataFrame after merging: 26109 rows, 58 columns

MANUAL COLUMN MERGING

Merging column group 1:
  Columns to merge:
    - 'vendor_id': 26060 non-null (99.8%) (PRIMARY)
    - 'vendor_id_number': 1 non-null (0.0%)
  → No additional values merged (primary column already complete)
  → Kept 'vendor_id' and dropped: vendor_id_number

Merging column group 2:
  Columns to merge:
    - 'fiscal_year_amount': 25270 non-null (96.8%) (PRIMARY)
    - 'fiscal_year_amounts': 5 non-null (0.0%)
  → Merged 5 additional value(s) from secondary column(s)
  → Kept 'fiscal_year_amount' and dropped: fiscal_year_amounts

Merging column group 3:
  Columns to m

In [84]:
# Display basic info
if not df.empty:
    print("DataFrame Info:")
    print(df.info())
    print("\n" + "="*60)
    print("First few rows:")
    display(df.head())
else:
    print("DataFrame is empty")

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26109 entries, 0 to 26108
Data columns (total 54 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   metadata_source_file            26108 non-null  object 
 1   metadata_page_number            26108 non-null  float64
 2   metadata_extracted_at           26108 non-null  object 
 3   metadata_model                  26108 non-null  object 
 4   eds_number                      26061 non-null  object 
 5   date_prepared                   26071 non-null  object 
 6   agency_name                     26078 non-null  object 
 7   agency_address                  26074 non-null  object 
 8   requisition_number              9872 non-null   object 
 9   agency_contact_name             26072 non-null  object 
 10  agency_contact_phone            26066 non-null  object 
 11  agency_contact_email            26059 non-null  object 
 12  vendor_name     

,metadata_source_file,metadata_page_number,metadata_extracted_at,metadata_model,eds_number,date_prepared,agency_name,agency_address,requisition_number,agency_contact_name,...,subvendor_minority,subvendor_women,subvendor_in_veteran,json_filename,error,raw_response,courier_name,courier_phone,courier_email,method_of_source_selection
0,../../../data/raw/_contracts/6643-002.pdf,1.0,2025-12-06T23:52:57.742179,claude-haiku-4-5-20251001,A70-7-8011,5/2/2007,Department of Health,"2 N. Meridian Street Indianapolis, IN 46204",None,Marcie L. Memmer,...,no,no,no_option,6643-002.json,NaN,NaN,NaN,NaN,NaN,NaN
1,../../../data/raw/_contracts/73772-001.pdf,1.0,2025-12-06T23:52:58.997258,claude-haiku-4-5-20251001,A249-14-320142A,2/7/2014,Indiana Dept of Transportation,Department Of Transportation Contract Admin Di...,None,Joel Myers,...,no,no,no,73772-001.json,NaN,NaN,NaN,NaN,NaN,NaN
2,../../../data/raw/_contracts/48197-003.pdf,1.0,2025-12-30T12:06:50.440661,claude-haiku-4-5-20251001,D12-0-018,8/8/2012,Department of Correction,Department of Correction Central Office 302 W ...,None,Andy Pritchard,...,no,no,no_option,48197-003.json,NaN,NaN,NaN,NaN,NaN,NaN
3,../../../data/raw/_contracts/85532-000.pdf,1.0,2025-12-06T23:53:00.824505,claude-haiku-4-5-20251001,T22-15-097,3/11/2015,State Racing Commission,"Horse Racing Commission, 1302 N MERIDIAN ST ST...",T22-S-010,Holly Newell,...,no,no,no,85532-000.json,NaN,NaN,NaN,NaN,NaN,NaN
4,../../../data/raw/_contracts/58037-000.pdf,1.0,2025-12-06T22:22:18.432347,claude-haiku-4-5-20251001,A58-2-12DL-0101,9/27/2011,Department of Education,"DOE-ACCOUNTING, 151 W OHIO ST, INDIANAPOLIS, I...",0000069117,Kim Payton,...,no,no,no_option,58037-000.json,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Data Quality Checks

In [105]:
df[df.error.notna()]

,metadata_source_file,metadata_page_number,metadata_extracted_at,metadata_model,eds_number,date_prepared,agency_name,agency_address,requisition_number,agency_contact_name,...,subvendor_minority,subvendor_women,subvendor_in_veteran,json_filename,error,raw_response,courier_name,courier_phone,courier_email,method_of_source_selection
187,../../../data/raw/_contracts/53011-001.pdf,1.0,2025-12-06T22:21:53.777202,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,no,no,None,53011-001.json,JSON parse failed,"{\n ""eds_number"": ""BB-1-822-11-H-2668"",\n ...",NaN,NaN,NaN,NaN
834,../../../data/raw/_contracts/26133-004.pdf,1.0,2025-12-05T12:01:04.281720,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,no,no,no_option,26133-004.json,ErrorResponse(error=APIErrorObject(message='In...,NaN,NaN,NaN,NaN,NaN
2171,../../../data/raw/_contracts/11014-003.pdf,1.0,2025-12-05T12:00:30.251859,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,no,no,no_option,11014-003.json,JSON parse failed,"{\n ""eds_number"": ""A249-6-320347"",\n ""da...",NaN,NaN,NaN,NaN
3694,../../../data/raw/_contracts/63492-000.pdf,1.0,2025-12-30T13:01:43.441149,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,no,no,no_option,63492-000.json,JSON parse failed,"{\n ""eds_number"": ""A55-3-63-13-20-1019"",\n ...",NaN,NaN,NaN,NaN
5630,../../../data/raw/_contracts/64978-001.pdf,1.0,2025-12-06T23:52:57.497762,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,yes,yes,None,64978-001.json,JSON parse failed,"{\n ""eds_number"": ""MD29-3-99-13-LF-1811"",\n...",NaN,NaN,NaN,NaN
5706,../../../data/raw/_contracts/86829-000.pdf,1.0,2025-12-06T23:53:01.045040,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,no,no,no,86829-000.json,JSON parse failed,"{\n ""eds_number"": ""A55-8-09-16-AM-0558"",\n ...",NaN,NaN,NaN,NaN
5716,../../../data/raw/_contracts/66569-000.pdf,1.0,2025-12-06T23:52:57.794670,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,no,no,no_option,66569-000.json,JSON parse failed,"{\n ""eds_number"": ""A345-3-49-13-77-1561"",\n...",NaN,NaN,NaN,NaN
6825,../../../data/raw/_contracts/93867-000.pdf,1.0,2025-12-06T23:53:01.818933,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,no,no,no,93867-000.json,JSON parse failed,"{\n ""eds_number"": ""A305-6-213"",\n ""date_...",NaN,NaN,NaN,NaN
8249,../../../data/raw/_contracts/65961-000.pdf,1.0,2025-12-06T23:52:57.740265,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,no,no,None,65961-000.json,JSON parse failed,"{\n ""eds_number"": ""A249-13-320046"",\n ""d...",NaN,NaN,NaN,NaN
9575,../../../data/raw/_contracts/30615-001.pdf,1.0,2025-12-06T10:28:35.247943,claude-haiku-4-5-20251001,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,30615-001.json,JSON parse failed,"{\n ""eds_number"": ""GA06-9-82-09-FA-0969"",\n...",NaN,NaN,NaN,NaN


In [87]:
if not df.empty:
    print("Data Quality Summary:")
    print("="*60)
    
    # Check for missing values
    print("\nMissing Values by Column:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Missing %': missing_pct
    })
    print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False))
    
    # Check key fields
    print("\n" + "="*60)
    print("Key Field Statistics:")
    
    key_fields = ['eds_number', 'vendor_name', 'agency_name', 'contract_types']
    for field in key_fields:
        if field in df.columns:
            non_null = df[field].notna().sum()
            pct = (non_null / len(df) * 100).round(1)
            print(f"  {field}: {non_null}/{len(df)} ({pct}%) populated")
else:
    print("DataFrame is empty")

Data Quality Summary:

Missing Values by Column:
                                Missing Count  Missing %
method_of_source_selection              26108      100.0
courier_email                           26106      100.0
courier_phone                           26106      100.0
courier_name                            26106      100.0
raw_response                            26087       99.9
error                                   26085       99.9
explanation_if_late                     20455       78.3
approver_names                          19169       73.4
signature_dates                         17039       65.3
requisition_number                      16237       62.2
primary_vendor_in_veteran                7677       29.4
subvendor_in_veteran                     4510       17.3
approval_dates                           2941       11.3
account_names                            1962        7.5
account_numbers                          1218        4.7
fiscal_year_amount                     

In [88]:
# Remove columns with 100% missing values
if not df.empty:
    print("\n" + "=" * 60)
    print("REMOVING EMPTY COLUMNS")
    print("=" * 60)
    
    # Identify columns with 100% missing values
    total_rows = len(df)
    empty_cols = []
    
    for col in df.columns:
        null_count = df[col].isnull().sum()
        if null_count == total_rows:
            empty_cols.append(col)
    
    if empty_cols:
        print(f"\nFound {len(empty_cols)} column(s) with 100% missing values:")
        for col in empty_cols:
            print(f"  - {col}")
        
        # Drop empty columns
        df = df.drop(columns=empty_cols)
        
        print(f"\nDropped {len(empty_cols)} empty column(s)")
        print(f"DataFrame now has {len(df.columns)} columns")
    else:
        print("\nNo columns with 100% missing values found.")
else:
    print("DataFrame is empty")


REMOVING EMPTY COLUMNS

No columns with 100% missing values found.


## 7. Save DataFrame (Optional)

In [107]:
# Uncomment to save the DataFrame to CSV
if not df.empty:
    output_path = jloc + 'data/intermediate_products/eds_claude_extraction_results.csv'
    df.to_csv(output_path, index=False)
    print(f"DataFrame saved to: {output_path}")
    print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
else:
    print("DataFrame is empty, not saving")

DataFrame saved to: ../../../data/intermediate_products/eds_claude_extraction_results.csv
Rows: 26109, Columns: 54
